In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio pydantic transformers accelerate bitsandbytes sentence-transformers
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 15.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.5 MB/s eta 

In [2]:
from fastapi import FastAPI
from pydantic import BaseModel
import unsloth
import uvicorn
from pyngrok import ngrok
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
import threading
import time
from getpass import getpass

# ==========================================
# 0. AUTENTIKASI HUGGING FACE
# ==========================================
HF_TOKEN = getpass("Masukkan HF token: ")
login(token=HF_TOKEN)

# ==========================================
# 1. LOAD AI MODELS
# ==========================================
print("Mulai mengunduh dan memuat BAAI/bge-m3...")
embedding_model = SentenceTransformer("BAAI/bge-m3")

print("Mulai mengunduh dan memuat Neiwawastaken/llama (4-bit)...")
model_id = "Neiwawastaken/legal-chatbot-llama3B-grpo"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1
)
print("Semua model AI berhasil dimuat ke GPU!")

# ==========================================
# 2. SETUP API & NGROK
# ==========================================
NGROK_AUTH_TOKEN = getpass("Ngrok Token :")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = FastAPI()

class ChatRequest(BaseModel):
    query: str
    retrieved_context: str = ""

@app.get("/")
def read_root():
    return {"status": "Legal RAG Backend is Running with Llama & BGE-m3!"}

@app.post("/api/chat")
def chat(request: ChatRequest):
    vector = embedding_model.encode(request.query)

    prompt = f"""Anda adalah asisten hukum ketenagakerjaan Indonesia. Jawab HANYA berdasarkan konteks berikut.
    Konteks: {request.retrieved_context}

    Pertanyaan: {request.query}
    Jawaban:"""

    hasil = llm(prompt)
    jawaban_final = hasil[0]['generated_text'].replace(prompt, "").strip()

    return {
        "query": request.query,
        "jawaban_model": jawaban_final,
        "dimensi_vektor": len(vector)
    }

# ==========================================
# 3. JALANKAN SERVER (DI THREAD TERPISAH)
# ==========================================
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

ngrok.kill()
public_url = ngrok.connect(8000)
print(f"🚀 API + AI Model online di: {public_url.public_url}/api/chat")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
print("✅ Server jalan di background thread — notebook tetap bisa dipakai untuk cell lain.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Masukkan HF token: ··········
Mulai mengunduh dan memuat BAAI/bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load Neiwawastaken/legal-chatbot-llama3B-grpo as a legacy tokenizer.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Semua model AI berhasil dimuat ke GPU!
Ngrok Token :··········
🚀 API + AI Model online di: https://unregardable-stephenie-nonmetrically.ngrok-free.dev/api/chat


INFO:     Started server process [453]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Server jalan di background thread — notebook tetap bisa dipakai untuk cell lain.
